# Emotional scoring
## *Words in Power, Words in Opposition* — Machine Learning for NLP
### Author: Salma El Aazdoudi

## Approach

I use three independent methods to score emotional intensity: **pyFeel** 
(lexicon-based, French), **thomasrenault/emotion** (fine-tuned DistilBERT, 
applied after translation), and **mDeBERTa** (zero-shot, multilingual). 
Each method is described in its own section below.

## 0. Setup

In [5]:
import numpy as np
import pandas as pd
import re
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

# ── numpy pickle patch (required by pyFeel) ───────────────────────────────
if not getattr(np.load, '_pickle_patched', False):
    _original_np_load = np.load
    def _patched_np_load(*args, **kwargs):
        kwargs.setdefault('allow_pickle', True)
        return _original_np_load(*args, **kwargs)
    _patched_np_load._pickle_patched = True
    np.load = _patched_np_load

# ── French tokeniser patch (required by pyFeel) ───────────────────────────
import nltk.tokenize as _nltk_tok
import nltk as _nltk
def _fr_tokenize(text, language='french'):
    return re.findall(r"[a-zàâçéèêëîïôûùüÿæœ'-]+", text.lower())
_nltk_tok.word_tokenize = _fr_tokenize
_nltk.word_tokenize     = _fr_tokenize

In [ ]:
# ── Load df_full ─────────────────────────────────────────────────────────
df_full = pd.read_csv("df_full.csv")

## 1. pyFeel (lexicon-based)

pyFeel (Abdaoui et al., 2017) is a French sentiment lexicon. It scores each text on seven dimensions : positivity and Paul Ekman's classification of emotions: `joy`, `fear`, `sadness`, `anger`, `surprise`, `disgust`. Each score is the proportion of emotion-tagged words in that category (bag-of-word approach)

The `positivity` score is a broad aggregate that dominates the output and is excluded from the composite. I add another pyFeel outcome is **`feel_intensity_nrc`** (the mean of the six emotion dimensions).


In [ ]:
!pip install git+https://github.com/AdilZouitine/pyFeel.git -q

from pyFeel import Feel
from tqdm import tqdm
import pandas as pd

EMOTION_COLS = ['positivity', 'joy', 'fear', 'sadness', 'angry', 'surprise', 'disgust']
NRC_COLS     = ['joy', 'fear', 'sadness', 'angry', 'surprise', 'disgust']

def score_pyfeel(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {e: float('nan') for e in EMOTION_COLS}
    try:
        return Feel(text).emotions()
    except Exception:
        return {e: float('nan') for e in EMOTION_COLS}

# Remplace progress_apply par une boucle tqdm standard
print(f"Scoring {len(df_full):,} texts with pyFeel...")
emotion_records = [
    score_pyfeel(text)
    for text in tqdm(df_full["text"].fillna("").tolist(), desc="Scoring with pyFeel")
]

df_feel = pd.DataFrame(emotion_records, index=df_full.index)
df_feel.columns = [f"feel_{col}" for col in df_feel.columns]
df_full = pd.concat([df_full, df_feel], axis=1)

feel_nrc_cols = [f"feel_{e}" for e in NRC_COLS]
df_full["feel_intensity_nrc"] = df_full[feel_nrc_cols].mean(axis=1)

print(f"\nScored {df_full[feel_nrc_cols].notna().all(axis=1).sum():,} / {len(df_full):,} manifestos")
print(f"\n=== pyFeel statistics ===")
print(df_full[feel_nrc_cols + ["feel_intensity_nrc"]].describe().round(4))
df_full.to_csv("df_full_with_pyfeel.csv", index=False)
print("Saved to df_full_with_pyfeel.csv")

## 2 — Transformer-based Emotion Model (thomasrenault/emotion)

As a second measure, we use **thomasrenault/emotion** (Renault, 2025), a DistilBERT model fine-tuned on ~200k US political texts (tweets, campaign speeches, and congressional floor speeches) annotated with GPT-4o-mini. It predicts 8 independent emotion intensities (sigmoid, range 0–1):

| Label | | Label | |
|-------|--|-------|--|
| `anger` | | `pride` | |
| `sadness` | | `joy` | |
| `fear` | | `gratitude` | |
| `disgust` | | `hope` | |

Unlike pyFeel, this model reads the full sentence and is sensitive to context and negation. It was designed specifically for research on emotion in political communication, making it the closest available model to our domain.

The pipeline is:
```
French text → chunked translation (facebook/nllb-200-1.3B) → emotion model → 8 scores```

### 2-1 — Translation

Texts are translated from French to English using [facebook/nllb-200-1.3B](https://huggingface.co/facebook/nllb-200-1.3B) (Meta AI). Because the professions de foi are long (mean: 1,081 tokens, max: 6,134 tokens), each text is split into overlapping 400-token chunks before translation and the results are concatenated. MarianMT models were initially tested but discarded due to hallucinations and truncation on long texts. The following cell takes 60 hours to run. 

In [ ]:
import torch
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── 1. Chargement ─────────────────────────────────────────────────────────
MODEL_ID     = "facebook/nllb-200-1.3B"
SRC_LANG     = "fra_Latn"
TGT_LANG     = "eng_Latn"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
ENG_TOKEN_ID = tokenizer_nllb.convert_tokens_to_ids(TGT_LANG)

# ── 2. Chunking + traduction ──────────────────────────────────────────────
CHUNK_SIZE = 400
OVERLAP    = 20

def chunk_text(text):
    tokenizer_nllb.src_lang = SRC_LANG
    tokens = tokenizer_nllb.encode(text, add_special_tokens=False)
    chunks, start = [], 0
    while start < len(tokens):
        end = min(start + CHUNK_SIZE, len(tokens))
        chunks.append(tokens[start:end])
        if end == len(tokens):
            break
        start += CHUNK_SIZE - OVERLAP
    return [tokenizer_nllb.decode(c) for c in chunks]

def translate_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    chunks = chunk_text(text)
    translated = []
    for chunk in chunks:
        try:
            tokenizer_nllb.src_lang = SRC_LANG
            inp = tokenizer_nllb(
                [chunk], return_tensors="pt",
                truncation=True, max_length=512,
            ).to(DEVICE)
            with torch.no_grad():
                out = model_nllb.generate(
                    **inp,
                    forced_bos_token_id  = ENG_TOKEN_ID,
                    max_new_tokens       = 512,
                    num_beams            = 4,
                    no_repeat_ngram_size = 4,
                    repetition_penalty   = 1.2,
                    early_stopping       = True,
                )
            translated.append(tokenizer_nllb.decode(out[0], skip_special_tokens=True))
        except Exception as e:
            print(f"  Chunk failed: {e}")
            translated.append("")
    return " ".join(translated)

# ── 3. Pipeline avec checkpoint ───────────────────────────────────────────
CHECKPOINT = Path("translations_nllb_checkpoint.csv")
SAVE_EVERY  = 50

if CHECKPOINT.exists():
    done = pd.read_csv(CHECKPOINT, index_col=0)["text_en"].dropna()
    done = {k: v for k, v in done.to_dict().items() if str(v).strip() != ""}
    print(f"Resuming — {len(done):,} already done.")
else:
    done = {}

texts = df_full["text"].fillna("").tolist()
ids   = df_full.index.tolist()

remaining_ids   = [i for i in ids if i not in done]
remaining_texts = [texts[ids.index(i)] for i in remaining_ids]
print(f"To translate: {len(remaining_texts):,} texts")

for j, (idx, text) in enumerate(tqdm(
    zip(remaining_ids, remaining_texts),
    total=len(remaining_texts), desc="Translating NLLB"
)):
    done[idx] = translate_text(text)
    if (j + 1) % SAVE_EVERY == 0:
        pd.DataFrame.from_dict(done, orient="index", columns=["text_en"])\
          .to_csv(CHECKPOINT)
        print(f"  Checkpoint saved — {len(done):,} done")

# ── 4. Sauvegarde finale ──────────────────────────────────────────────────
pd.DataFrame.from_dict(done, orient="index", columns=["text_en"])\
  .to_csv(CHECKPOINT)

df_full["text_en"] = pd.Series(done)
df_full.to_csv("df_full_translated_nllb.csv", index=False)

print(f"\nDone — {len(done):,} texts translated.")
print(f"Example FR : {texts[0][:200]}")
print(f"Example EN : {df_full['text_en'].iloc[0][:200]}")

### 2-2 Emotional Scoring with Bert

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── 1. Chargement du modèle ───────────────────────────────────────────────
EMOTION_MODEL_ID = "thomasrenault/emotion"
EMOTIONS         = ["anger", "sadness", "fear", "disgust", "pride", "joy", "gratitude", "hope"]
NRC_SHARED       = ["anger", "sadness", "fear", "disgust", "joy"]

print("Loading emotion model...")
tokenizer_emo = AutoTokenizer.from_pretrained(EMOTION_MODEL_ID)
model_emo     = AutoModelForSequenceClassification.from_pretrained(EMOTION_MODEL_ID)
model_emo.eval()
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
model_emo = model_emo.to(DEVICE)
print(f"Model loaded on {DEVICE}.")

# ── 2. Scoring par batch ──────────────────────────────────────────────────
BATCH_SIZE = 32

def score_batch(texts: list) -> list:
    results = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Emotion scoring"):
        batch = texts[i:i+BATCH_SIZE]
        try:
            enc = tokenizer_emo(
                batch,
                return_tensors = "pt",
                truncation     = True,
                max_length     = 512,
                padding        = True,
            ).to(DEVICE)
            with torch.no_grad():
                probs = torch.sigmoid(model_emo(**enc).logits).tolist()
            for p in probs:
                results.append(dict(zip(EMOTIONS, p)))
        except Exception as e:
            print(f"  Batch error at {i}: {e} — fallback one by one")
            for text in batch:
                try:
                    enc = tokenizer_emo(
                        [text], return_tensors="pt",
                        truncation=True, max_length=512,
                    ).to(DEVICE)
                    with torch.no_grad():
                        p = torch.sigmoid(model_emo(**enc).logits).squeeze().tolist()
                    results.append(dict(zip(EMOTIONS, p)))
                except Exception:
                    results.append({e: float("nan") for e in EMOTIONS})
    return results

# ── 3. Application sur les textes traduits ────────────────────────────────
texts_en = df_full["text_en"].fillna("").tolist()
print(f"Scoring {len(texts_en):,} texts...")
all_scores = score_batch(texts_en)

# ── 4. Intégration dans df_full ───────────────────────────────────────────
df_tr = pd.DataFrame(all_scores, index=df_full.index)
df_tr.columns = [f"tr_{col}" for col in df_tr.columns]
df_full = pd.concat([df_full, df_tr], axis=1)

tr_nrc_cols = [f"tr_{e}" for e in NRC_SHARED]
df_full["tr_intensity_nrc"] = df_full[tr_nrc_cols].mean(axis=1)

# ── 5. Sauvegarde ─────────────────────────────────────────────────────────
df_full.to_csv("df_full_with_emotions.csv", index=False)

print("\nDone.")
print(f"\n=== thomasrenault/emotion statistics ===")
print(df_full[tr_nrc_cols + ["tr_intensity_nrc"]].describe().round(4)

## 3 - Zero-Shot Emotion Scoring (mDeBERTa)

I use [MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7](https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7), a multilingual model applied directly to the original French text. For each emotion, the model evaluates whether the hypothesis "This text expresses [e]" is entailed by the input — scores are computed independently per label, allowing multiple emotions to score highly simultaneously. The five labels are anger, fear, sadness, disgust, and joy.

In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import pipeline

# ── 1. Chargement du modèle ───────────────────────────────────────────────
MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
DEVICE   = 0 if torch.cuda.is_available() else -1

print("Loading mDeBERTa zero-shot model...")
classifier = pipeline(
    "zero-shot-classification",
    model  = MODEL_ID,
    device = DEVICE,
)
print(f"Model loaded on {'cuda' if DEVICE == 0 else 'cpu'}.")

# ── 2. Labels émotionnels ─────────────────────────────────────────────────
EMOTION_LABELS = ["anger", "fear", "sadness", "disgust", "joy"]

# ── 3. Fonction de scoring ────────────────────────────────────────────────
def score_zero_shot(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {f"zs_{e}": np.nan for e in EMOTION_LABELS} | {"zs_dominant": None}
    try:
        result = classifier(
            text[:2000],  # ~512 tokens approximatif
            candidate_labels    = EMOTION_LABELS,
            multi_label         = True,
            hypothesis_template = "This text expresses {}.",
        )
        scores   = dict(zip(result["labels"], result["scores"]))
        dominant = max(scores, key=scores.get)
        return {f"zs_{e}": scores.get(e, np.nan) for e in EMOTION_LABELS} | {"zs_dominant": dominant}
    except Exception as e:
        print(f"  Error: {e}")
        return {f"zs_{e}": np.nan for e in EMOTION_LABELS} | {"zs_dominant": None}

# ── 4. Application sur le corpus ──────────────────────────────────────────
# Texte français directement — pas de traduction nécessaire
texts = df_full["text"].fillna("").tolist()
ids   = df_full.index.tolist()

print(f"Scoring {len(texts):,} texts...")
all_scores = []
for text in tqdm(texts, desc="Zero-shot scoring"):
    all_scores.append(score_zero_shot(text))

# ── 5. Intégration dans df_full ───────────────────────────────────────────
df_zs = pd.DataFrame(all_scores, index=df_full.index)
df_full = pd.concat([df_full, df_zs], axis=1)

zs_nrc_cols = [f"zs_{e}" for e in EMOTION_LABELS]
df_full["zs_intensity_nrc"] = df_full[zs_nrc_cols].mean(axis=1)

# ── 6. Sauvegarde ─────────────────────────────────────────────────────────
df_full.to_csv("df_full_with_zeroshot.csv", index=False)

print("\nDone.")
print(f"\n=== mDeBERTa zero-shot statistics ===")
print(df_full[zs_nrc_cols + ["zs_intensity_nrc"]].describe().round(4))
print(f"\nDominant emotion distribution:")
print(df_full["zs_dominant"].value_counts())

## 4- Comparison across the three models 

### 4-1 Do the models agree on the dominant emotion ?

In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("df_full_merged.csv")

# ── Colonnes par modèle ───────────────────────────────────────────────────
TR_EMOTIONS   = {"tr_anger": "anger", "tr_sadness": "sadness", "tr_fear": "fear",
                 "tr_disgust": "disgust", "tr_joy": "joy"}
ZS_EMOTIONS   = {"zs_anger": "anger", "zs_fear": "fear", "zs_sadness": "sadness",
                 "zs_disgust": "disgust", "zs_joy": "joy"}
FEEL_EMOTIONS = {"feel_angry": "anger", "feel_fear": "fear", "feel_sadness": "sadness",
                 "feel_disgust": "disgust", "feel_joy": "joy"}

def get_dominant(row, col_map):
    scores = {label: row[col] for col, label in col_map.items()
              if col in row.index and pd.notna(row[col])}
    if not scores:
        return None
    return max(scores, key=scores.get)

# ── Calculer l'émotion dominante pour chaque modèle ──────────────────────
df["dom_tr"]   = df.apply(lambda r: get_dominant(r, TR_EMOTIONS),   axis=1)
df["dom_zs"]   = df.apply(lambda r: get_dominant(r, ZS_EMOTIONS),   axis=1)
df["dom_feel"] = df.apply(lambda r: get_dominant(r, FEEL_EMOTIONS), axis=1)

# Garder les lignes où les 3 modèles ont un résultat
df_valid = df[["dom_tr", "dom_zs", "dom_feel", "bloc"]].dropna()
n = len(df_valid)
print(f"Textes analysés : {n:,}\n")

# ── Taux d'accord 3/3 ────────────────────────────────────────────────────
agree_all = (
    (df_valid["dom_tr"] == df_valid["dom_zs"]) &
    (df_valid["dom_zs"] == df_valid["dom_feel"])
)
print(f"Accord 3/3 (tous d'accord)   : {agree_all.sum():,} / {n:,} ({agree_all.mean()*100:.1f}%)")

# ── Taux d'accord 2/3 ────────────────────────────────────────────────────
agree_tr_zs   = df_valid["dom_tr"]   == df_valid["dom_zs"]
agree_tr_feel = df_valid["dom_tr"]   == df_valid["dom_feel"]
agree_zs_feel = df_valid["dom_zs"]   == df_valid["dom_feel"]
agree_any2 = agree_tr_zs | agree_tr_feel | agree_zs_feel

print(f"Accord 2/3 (au moins 2/3)    : {agree_any2.sum():,} / {n:,} ({agree_any2.mean()*100:.1f}%)")
print(f"  thomasrenault + mDeBERTa   : {agree_tr_zs.sum():,} ({agree_tr_zs.mean()*100:.1f}%)")
print(f"  thomasrenault + pyFeel     : {agree_tr_feel.sum():,} ({agree_tr_feel.mean()*100:.1f}%)")
print(f"  mDeBERTa + pyFeel          : {agree_zs_feel.sum():,} ({agree_zs_feel.mean()*100:.1f}%)")

# ── Désaccord total ───────────────────────────────────────────────────────
disagree_all = ~agree_any2
print(f"\nDésaccord total (0/3)        : {disagree_all.sum():,} / {n:,} ({disagree_all.mean()*100:.1f}%)")

# ── Distribution des émotions dominantes par modèle ──────────────────────
print(f"\n=== Distribution émotions dominantes ===")
for col, name in [("dom_tr", "thomasrenault"), ("dom_zs", "mDeBERTa"), ("dom_feel", "pyFeel")]:
    print(f"\n{name}:")
    print(df_valid[col].value_counts().to_string())

# ── Accord par bloc ───────────────────────────────────────────────────────
print(f"\n=== Accord 3/3 par bloc ===")
df_valid["agree_all"] = agree_all
print(df_valid.groupby("bloc")["agree_all"].agg(["sum", "mean"]).round(3).to_string())

Textes analysés : 6,446

Accord 3/3 (tous d'accord)   : 175 / 6,446 (2.7%)
Accord 2/3 (au moins 2/3)    : 3,448 / 6,446 (53.5%)
  thomasrenault + mDeBERTa   : 1,273 (19.7%)
  thomasrenault + pyFeel     : 1,448 (22.5%)
  mDeBERTa + pyFeel          : 1,077 (16.7%)

Désaccord total (0/3)        : 2,998 / 6,446 (46.5%)

=== Distribution émotions dominantes ===

thomasrenault:
dom_tr
joy        3442
anger      2266
fear        488
sadness     249
disgust       1

mDeBERTa:
dom_zs
sadness    3657
anger      1590
joy         500
disgust     463
fear        236

pyFeel:
dom_feel
anger      3747
fear       2014
sadness     597
joy          52
disgust      36

=== Accord 3/3 par bloc ===
           sum   mean
bloc                 
ecologist    4  0.007
far_left     5  0.006
far_right   39  0.049
left        66  0.028
right       61  0.032


#### Cross-Model Agreement on Dominant Emotion

Agreement across the three methods is low: all three assign the same dominant emotion 
in only 2.7% of texts (175/6,446), and at least two agree in 53.5% of cases. 

This is not surprising given how differently the three models work, but it has 
implications for interpretation.

###### Pairwise agreement

thomasrenault and pyFeel agree most often (22.5%), followed by thomasrenault and 
mDeBERTa (19.7%), with mDeBERTa and pyFeel agreeing least (16.7%). The fact that 
the two most methodologically different models (a fine-tuned BERT and a word-counting 
lexicon) agree more often than either does with mDeBERTa suggests that mDeBERTa is 
the most discrepant of the three.

###### What the distributions reveal

The dominant emotion distributions tell a clear story about each model's bias:

- **thomasrenault** assigns joy to 53% of texts : the most common label by far. 
This likely reflects the generally aspirational tone of electoral manifestos, which 
the model has learned to associate with positive affect.
- **mDeBERTa** assigns sadness to 57% of texts.
- **pyFeel** assigns anger to 58% of texts. Since most manifestos contain some 
critical or adversarial language, the bag-of-words model defaults to anger whenever 
negative words outnumber positive ones.

Each model has a strong default label: joy for thomasrenault, sadness for mDeBERTa, 
anger for pyFeel.
    
###### What this means for the analysis (and the research questions that build on this scoring)

Low agreement on dominant emotion does not mean the models are useless, it means 
they are measuring different things. The more robust comparison is not which emotion 
dominates, but whether the models agree on *intensity*: do the same candidates score 
high across all three methods? 

### 4-2 Manual Inspection 

In [11]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import re

df = pd.read_csv("df_full_merged.csv")

def clean_text(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'Sciences?\s*Po\s*/?\s*f(?:onds?|unds?)\s*CEVI\w*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Sciences?\s*Po\s*/?\s*CEVI\w*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# ── Colonnes par modèle ───────────────────────────────────────────────────
TR_EMOTIONS   = {"tr_anger": "anger", "tr_sadness": "sadness", "tr_fear": "fear",
                 "tr_disgust": "disgust", "tr_joy": "joy"}
ZS_EMOTIONS   = {"zs_anger": "anger", "zs_fear": "fear", "zs_sadness": "sadness",
                 "zs_disgust": "disgust", "zs_joy": "joy"}
FEEL_EMOTIONS = {"feel_angry": "anger", "feel_fear": "fear", "feel_sadness": "sadness",
                 "feel_disgust": "disgust", "feel_joy": "joy"}

EMOTION_EMOJI = {"anger": "😠", "fear": "😨", "sadness": "😢",
                 "disgust": "🤢", "joy": "😊", "surprise": "😲",
                 "positivity": "✨"}

bloc_colors = {
    "far_left": "#d73027", "left": "#fc8d59",
    "ecologist": "#4dac26", "right": "#91bfdb", "far_right": "#2166ac"
}

def get_dominant(row, col_map):
    scores = {label: row[col] for col, label in col_map.items() if col in row.index and pd.notna(row[col])}
    if not scores:
        return "—", 0
    dominant = max(scores, key=scores.get)
    return dominant, scores[dominant]

def display_comparison_table(df, n=10, random_state=42):
    needed = list(TR_EMOTIONS.keys()) + list(ZS_EMOTIONS.keys()) + list(FEEL_EMOTIONS.keys()) + ["text", "bloc"]
    available = [c for c in needed if c in df.columns]
    sample = df[available].dropna(subset=list(TR_EMOTIONS.keys())).sample(n, random_state=random_state)

    html = """
    <style>
        .cmp-table { width:100%; border-collapse:collapse; font-family:Arial, sans-serif; font-size:12px; }
        .cmp-table th { background:#2c3e50; color:white; padding:8px; text-align:center; }
        .cmp-table td { padding:8px; vertical-align:middle; border-bottom:1px solid #eee; }
        .cmp-table tr:hover { background:#f5f5f5; }
        .full-text { font-size:11px; color:#333; line-height:1.5; white-space:pre-wrap; max-height:150px; overflow-y:auto; width:280px; }
        details summary { cursor:pointer; font-size:11px; color:#2980b9; }
        .agree { background:#d5f5e3; border-radius:4px; padding:2px 4px; }
        .disagree { background:#fde8e8; border-radius:4px; padding:2px 4px; }
    </style>
    <table class='cmp-table'>
        <tr>
            <th style='width:28%'>Text</th>
            <th>Bloc</th>
            <th>pyFeel</th>
            <th>thomasrenault</th>
            <th>mDeBERTa</th>
            <th>Agreement</th>
        </tr>
    """

    for _, row in sample.iterrows():
        # Dominant emotion per model
        tr_dom,   tr_val   = get_dominant(row, TR_EMOTIONS)
        zs_dom,   zs_val   = get_dominant(row, ZS_EMOTIONS)
        feel_dom, feel_val = get_dominant(row, FEEL_EMOTIONS)

        # Agreement check
        all_three = [tr_dom, zs_dom, feel_dom]
        if len(set(all_three)) == 1:
            agreement = "<span class='agree'>✓ All agree</span>"
        elif len(set(all_three)) == 2:
            agreement = "<span>~ Partial</span>"
        else:
            agreement = "<span class='disagree'>✗ Disagree</span>"

        bloc = row.get("bloc", "")
        bloc_color = bloc_colors.get(bloc, "#999")
        bloc_badge = f"<span style='background:{bloc_color};color:white;padding:2px 6px;border-radius:8px;font-size:10px;'>{bloc}</span>"

        full_text = clean_text(str(row.get("text", ""))).replace("<", "&lt;").replace(">", "&gt;")
        preview   = full_text[:80] + "..."
        text_cell = f"<details><summary>{preview}</summary><div class='full-text'>{full_text}</div></details>"

        def fmt(dom, val):
            emoji = EMOTION_EMOJI.get(dom, "")
            return f"<b>{emoji} {dom}</b><br><small>{val:.2f}</small>"

        html += f"""
        <tr>
            <td>{text_cell}</td>
            <td style='text-align:center'>{bloc_badge}</td>
            <td style='text-align:center'>{fmt(feel_dom, feel_val)}</td>
            <td style='text-align:center'>{fmt(tr_dom, tr_val)}</td>
            <td style='text-align:center'>{fmt(zs_dom, zs_val)}</td>
            <td style='text-align:center'>{agreement}</td>
        </tr>
        """

    html += "</table>"
    display(HTML(html))

display_comparison_table(df, n=10)

In [12]:
def save_comparison_table(df, path="comparison_table.html", n=10, random_state=42):
    needed = list(TR_EMOTIONS.keys()) + list(ZS_EMOTIONS.keys()) + list(FEEL_EMOTIONS.keys()) + ["text", "bloc"]
    available = [c for c in needed if c in df.columns]
    sample = df[available].dropna(subset=list(TR_EMOTIONS.keys())).sample(n, random_state=random_state)

    rows_html = ""
    for _, row in sample.iterrows():
        tr_dom,   tr_val   = get_dominant(row, TR_EMOTIONS)
        zs_dom,   zs_val   = get_dominant(row, ZS_EMOTIONS)
        feel_dom, feel_val = get_dominant(row, FEEL_EMOTIONS)

        all_three = [tr_dom, zs_dom, feel_dom]
        if len(set(all_three)) == 1:
            agreement = "<span class='agree'>✓ All agree</span>"
        elif len(set(all_three)) == 2:
            agreement = "<span>~ Partial</span>"
        else:
            agreement = "<span class='disagree'>✗ Disagree</span>"

        bloc       = row.get("bloc", "")
        bloc_color = bloc_colors.get(bloc, "#999")
        bloc_badge = f"<span style='background:{bloc_color};color:white;padding:2px 6px;border-radius:8px;font-size:10px;'>{bloc}</span>"

        full_text = clean_text(str(row.get("text", ""))).replace("<", "&lt;").replace(">", "&gt;")
        preview   = full_text[:80] + "..."
        text_cell = f"<details><summary>{preview}</summary><div class='full-text'>{full_text}</div></details>"

        def fmt(dom, val):
            emoji = EMOTION_EMOJI.get(dom, "")
            return f"<b>{emoji} {dom}</b><br><small>{val:.2f}</small>"

        rows_html += f"""
        <tr>
            <td>{text_cell}</td>
            <td style='text-align:center'>{bloc_badge}</td>
            <td style='text-align:center'>{fmt(feel_dom, feel_val)}</td>
            <td style='text-align:center'>{fmt(tr_dom, tr_val)}</td>
            <td style='text-align:center'>{fmt(zs_dom, zs_val)}</td>
            <td style='text-align:center'>{agreement}</td>
        </tr>"""

    full_html = f"""<!DOCTYPE html>
<html><head><meta charset='utf-8'>
<title>Emotion Model Comparison</title>
<style>
    body {{ font-family: Arial, sans-serif; padding: 20px; }}
    .cmp-table {{ width:100%; border-collapse:collapse; font-size:12px; }}
    .cmp-table th {{ background:#2c3e50; color:white; padding:8px; text-align:center; }}
    .cmp-table td {{ padding:8px; vertical-align:middle; border-bottom:1px solid #eee; }}
    .cmp-table tr:hover {{ background:#f5f5f5; }}
    .full-text {{ font-size:11px; color:#333; line-height:1.5; white-space:pre-wrap; max-height:150px; overflow-y:auto; width:280px; }}
    details summary {{ cursor:pointer; font-size:11px; color:#2980b9; }}
    .agree {{ background:#d5f5e3; border-radius:4px; padding:2px 4px; }}
    .disagree {{ background:#fde8e8; border-radius:4px; padding:2px 4px; }}
</style>
</head><body>
<h2>Emotion Model Comparison — pyFeel vs thomasrenault vs mDeBERTa</h2>
<table class='cmp-table'>
    <tr>
        <th style='width:28%'>Text</th>
        <th>Bloc</th>
        <th>pyFeel</th>
        <th>thomasrenault</th>
        <th>mDeBERTa</th>
        <th>Agreement</th>
    </tr>
    {rows_html}
</table>
</body></html>"""

    with open(path, "w", encoding="utf-8") as f:
        f.write(full_html)
    print(f"Saved to {path}")

save_comparison_table(df, path="comparison_table.html", n=10)

Saved to comparison_table.html


## Cross-Model Comparison: Manual Inspection of 10 Manifestos

The table below compares the dominant emotion assigned by each model on the same 
10 texts, read in full to assess face validity.
---
**1. Mouvement des Réformateurs — right, Paris 12e, 1993**
*pyFeel: fear (0.11) | thomasrenault: joy (0.47) | mDeBERTa: sadness (0.99)*
**Outcome: thomasrenault closest, full disagreement.**

The text opens with civic pride (*"La France est un beau pays"*) and calls for 
solidarity and reform. The tone is constructive and moderate throughout (21 policy 
proposals, no hostility). thomasrenault's assignment of joy is defensible. pyFeel's 
fear likely reflects words like *"haine"*, *"méchanceté"*, *"sécurité menacée"* in 
the first paragraph without reading the overall optimistic register. mDeBERTa's 
saturation at sadness 0.99 is implausible for a reform manifesto.
---

**2. Mouvement des Citoyens (Gisèle Sebag) — left, Paris 8e, 1993**
*pyFeel: anger (0.08) | thomasrenault: fear (0.38) | mDeBERTa: fear (1.00)*
**Outcome: thomasrenault and mDeBERTa agree on fear, partial agreement.**

The text is explicitly anxious — *"une société brutale, injuste"*, *"notre sort est 
en jeu"*, concern about youth unemployment. Fear is a reasonable dominant emotion. 
pyFeel picks up anger which is also present but secondary. mDeBERTa's score of 1.00 
is inflated but the direction is correct. The two BERT models agree here, which 
increases confidence.

---
**3. Front National (Roselyne Vialles) — far_right, Hérault, 1993**
*pyFeel: fear (0.16) | thomasrenault: anger (0.66) | mDeBERTa: anger (1.00)*
**Outcome: thomasrenault and mDeBERTa correct, pyFeel wrong.**

The manifesto is built around explicit accusation and denunciation — *"TOUS 
RESPONSABLES, TOUS COUPABLES"*, calls to reinstate the death penalty, lists of 
threats. Anger is unambiguously the dominant register. pyFeel assigns fear at only 
0.16, which is a clear failure, the bag-of-words model picks up threat vocabulary 
(*"insécurité"*, *"immigration"*) as fear rather than as the aggressive framing it 
actually is. Both BERT models correctly identify anger.
---
**4. Parti Communiste (Alain Feuchot) — left, Ardèche, 1993**
*pyFeel: anger (0.21) | thomasrenault: anger (0.71) | mDeBERTa: sadness (1.00)*
**Outcome: pyFeel and thomasrenault correct, mDeBERTa wrong.**

The text is combative throughout — *"quel écœurant spectacle"*, *"agression contre 
vos conditions d'existence"*, explicit calls to mobilise against the right. Anger is 
the dominant register, not sadness. Both pyFeel and thomasrenault get this right. 
mDeBERTa's sadness assignment likely reflects the vocabulary of hardship (chômage, 
austérité, injustice) without capturing the confrontational framing.

---
**5. Parti Communiste (Marie-George Buffet) — left, Hauts-de-Seine, 1993**
*pyFeel: anger (0.18) | thomasrenault: anger (0.51) | mDeBERTa: sadness (1.00)*
**Outcome: pyFeel and thomasrenault correct, mDeBERTa wrong.**
Similar to Feuchot but more programmatic. The headline reads *"POUR EXPRIMER TOUT 
VOTRE MÉCONTENTEMENT"* — anger is clearly the intended register. pyFeel and 
thomasrenault both assign anger. mDeBERTa again confuses the hardship vocabulary 
with sadness. Notably, thomasrenault distinguishes Buffet (0.51) from Feuchot (0.71) 
, which is a meaningful difference given that Buffet's text is more constructive. mDeBERTa 
gives both 1.00, losing that distinction entirely.

---
**6. Independent candidate (François Mazoyer) — right, Loire, 1993**
*pyFeel: anger (0.10) | thomasrenault: sadness (0.37) | mDeBERTa: sadness (0.95)*
**Outcome: thomasrenault and mDeBERTa agree on sadness, plausible.**

A locally-focused manifesto from a mayor (infrastructure, employment, local roads). 
The opening acknowledges *"une crise économique, sociale, politique et morale"* which 
explains the sadness assignment. There is no anger or joy (the text is purely 
pragmatic). thomasrenault and mDeBERTa agree on sadness, which is reasonable. pyFeel's 
anger (0.10) is the lowest score in the sample, correctly signalling low intensity, 
but the direction is off. This is one of the cleaner agreements in the sample.

---
**7. Parti Socialiste (Pierre Jacquemin) — left, Val-de-Marne, 1981**
*pyFeel: fear (0.16) | thomasrenault: joy (0.62) | mDeBERTa: anger (0.29)*
**Outcome: thomasrenault correct, full disagreement.**

Written immediately after Mitterrand's election — *"une explosion d'espoir"*, *"la 
victoire de la FRANCE qui travaille, qui crée, qui rêve"*. This is the most 
unambiguously joyful text in the sample. thomasrenault correctly identifies joy at 
0.62. pyFeel assigns fear (probably reacting to anti-right passages) at the end 
(*"cette droite qui a fait faillite"*). mDeBERTa assigns anger at only 0.29, its 
lowest score across the sample, which at least signals low intensity even if the 
direction is wrong. This case strongly favours the fine-tuned model over both 
alternatives.

---
**8. Ecologists' Entente (Alain Bertolino) — ecologist, Gard, 1993**
*pyFeel: anger (0.13) | thomasrenault: joy (0.47) | mDeBERTa: anger (0.87)*
**Outcome: thomasrenault correct, partial disagreement.**

The manifesto is explicitly constructive — *"donner une vraie chance à l'écologie"*, 
*"solutions humaines, audacieuses"*, a call to *"ouvrir les portes de l'Assemblée"*. 
Joy is the right dominant emotion. pyFeel and mDeBERTa both assign anger, likely 
reacting to passages about pollution and economic recession (*"le chômage, la 
récession économique"*). thomasrenault alone captures the positive, forward-looking 
register. This is another case where the fine-tuned model outperforms the other two.

---
**9. RPR-UDF (Jean-Pierre Delalande) — right, Val-d'Oise, 1993**
*pyFeel: fear (0.14) | thomasrenault: fear (0.36) | mDeBERTa: sadness (1.00)*
**Verdict: pyFeel and thomasrenault agree on fear, plausible.**

The manifesto diagnoses national decline — *"la situation de notre pays s'est 
fortement détériorée"*, *"pessimisme préoccupant"*, *"perte de confiance"*. Fear is 
a reasonable dominant emotion for an anxious diagnostic framing. pyFeel and 
thomasrenault agree, which is reassuring. mDeBERTa's saturation at sadness 1.00 is 
implausible — though sadness is adjacent to fear, the extreme score loses all 
nuance. thomasrenault's 0.36 is more credible.
---

**10. Independent candidate (Jehan Huck) — right, Nièvre, 1993**
*pyFeel: anger (0.14) | thomasrenault: anger (0.67) | mDeBERTa: sadness (0.98)*
**Outcome: pyFeel and thomasrenault correct, mDeBERTa wrong.**

A personal grievance manifesto (18 property seizures, corrupt officials, the 
*"pogrome"* against the candidate). The closing line is *"VIVE LA FRANCE 
RÉPUBLICAINE"*. Anger is unambiguous throughout. pyFeel and thomasrenault correctly 
identify anger. mDeBERTa assigns sadness, likely reacting to the victimhood 
vocabulary rather than the combative, accusatory framing.


## Conclusion

Three methods were tested. They do not agree much — full agreement across all three 
occurs in only 2.7% of texts — but the disagreements are informative.

**pyFeel** defaults to anger on most texts because adversarial language is 
structurally common in electoral manifestos. It is measuring emotional vocabulary 
density, not emotional register. A manifesto that opens with hope and ends with a 
policy list will score as angry if it mentions unemployment twice. Its results need 
to be read carefully, but it remains useful as a robustness check on the anger and 
fear dimensions.

**mDeBERTa** cannot distinguish between a text that *talks about* hardship and a 
text that *expresses* sadness. A Front National tract listing unemployment figures 
is making an accusation, not lamenting. The model does not make that distinction, 
which is why it assigns sadness to 57% of the corpus. Its individual-level 
assignments are not reliable on this corpus.

**thomasrenault** is the most useful of the three. It correctly identifies joy in 
positive texts, anger in combative ones, and its scores are calibrated enough to 
distinguish between candidates of similar ideology. The translation step adds noise 
and the American training data does not always transfer cleanly to French legislative 
rhetoric from the 1980s, but these are manageable limitations.

### Measurement strategy across research questions

The three models are not used symmetrically. For **RQ1** (ideology and emotional 
intensity), I use the intensity scores from all three methods. The U-shape hypothesis 
is the most central claim in this project — convergence across three independent 
measures with different failure modes strengthens the robustness argument.

For **RQ2** (government vs opposition) and **RQ3** (economic context), I rely on 
**thomasrenault** only. These questions are about directional effects on specific 
emotions — in particular whether incumbents use more *positive* language. pyFeel 
assigns joy as the dominant emotion in only 52 texts out of 6,446 and mDeBERTa has 
the saturation problem documented above. thomasrenault assigns joy to 3,442 texts, 
providing enough variance to test these hypotheses meaningfully.